In [ ]:
# 11_paper_figures cell 1: config, shared style, helpers. Prints inputs for Fig 1.
import os, sys, io, re, json, ast, shutil, hashlib, pickle
import numpy as np, pandas as pd
import matplotlib as mpl; mpl.use("Agg"); import matplotlib.pyplot as plt, matplotlib.text as mtext
from PIL import Image
assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED is not '0': wrong kernel, STOP"
SEED = 42; R = os.environ["CREDIT_BN_ROOT"]
sys.path.insert(0, R + "/shared/src"); import safe_save as ss
W = R + "/Submissions/4. Article/WRITEUP/"; FIGW = W + "figures/"; os.makedirs(FIGW, exist_ok=True)
LEDP = W + "ledgers/numbers_ledger_v3.csv"; WIDTH = 6.83
STYLE = {"BN": dict(color="#0072B2", marker="o", ls="-"), "LR": dict(color="#D55E00", marker="s", ls="--"),
         "RF": dict(color="#009E73", marker="^", ls=":"), "trivial": dict(color="0.45", ls="-.")}
BANNED = re.compile(r"decision|significan|robust|invisible", re.I)
def apply_style():
    mpl.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
        "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "axes.linewidth": 0.6,
        "lines.linewidth": 1.2, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
def check_style(fig):
    assert abs(fig.get_figwidth() - WIDTH) < 1e-6, "figure width is not 6.83 in"
    assert all(ax.get_title() == "" for ax in fig.axes) and fig._suptitle is None, "title inside figure"
    ts = [t for t in fig.findobj(mtext.Text) if t.get_visible() and t.get_text().strip()]
    bad = [(t.get_text()[:30], t.get_fontsize()) for t in ts if t.get_fontsize() < 7 - 1e-9]; assert not bad, "text under 7 pt: %s" % bad[:5]
    w = [t.get_text() for t in ts if BANNED.search(t.get_text())]; assert not w, "banned wording: %s" % w
    return min(t.get_fontsize() for t in ts)
def load_led(): return pd.read_csv(LEDP, dtype=str, keep_default_na=False)
def map_ids(rows):
    led = load_led()
    for r in rows:
        m = led[(led.source_file == r["source_file"]) & (led.source_locator == r["source_locator"])]
        r["ledger_id"] = m.ledger_id.iloc[0] if len(m) else "NOT IN LEDGER"
    return rows
def append_g10(rows, stage, quantity_fn, kind="p4"):
    led = load_led(); new = []; nid = max([int(i.split("-")[2]) for i in led.ledger_id if i.startswith("N-FIG-")] + [0])
    for r in rows:
        if r["ledger_id"] != "NOT IN LEDGER": continue
        nid += 1; sh = hashlib.sha256(open(R + "/" + r["source_file"], "rb").read()).hexdigest()[:12]; x = {c: "" for c in led.columns}
        x.update(ledger_id="N-FIG-%03d" % nid, stage=stage, quantity=quantity_fn(r), value=repr(float(r["value"])), display=r["display"], source_file=r["source_file"], source_locator=r["source_locator"],
                 source_sha12=sh, derived="N", notes="gap G10: number printed in a paper figure, not previously in the ledger", display_rule=kind); new.append(x); r["ledger_id"] = x["ledger_id"]
    if new:
        out = pd.concat([led, pd.DataFrame(new)], ignore_index=True); assert out.ledger_id.is_unique; out.to_csv(LEDP, index=False)
    return len(new)
def save_fig(fig, name, rows):
    mn = check_style(fig); paths = {e: R + "/selection/figures/%s.%s" % (name, e) for e in ("png", "pdf")}
    for p in paths.values(): ss.check_path(p)
    npth = FIGW + name + "_numbers.csv"; assert not os.path.exists(npth), "numbers file exists"
    bufs = {}
    for e in paths: b = io.BytesIO(); fig.savefig(b, format=e, dpi=300); bufs[e] = b.getvalue()
    for e, p in paths.items():
        with open(ss.check_path(p), "xb") as f: f.write(bufs[e])
        shutil.copy2(p, FIGW + os.path.basename(p))
    pd.DataFrame(rows, columns=["label", "display", "ledger_id", "source_file", "source_locator", "value"]).to_csv(npth, index=False)
    w, h = fig.get_size_inches(); plt.close(fig)
    print("%s: %.2f x %.2f in | smallest font %.1f pt (asserted >= 7) | numbers: %d (NOT IN LEDGER: %d)" % (name, w, h, mn, len(rows), sum(r["ledger_id"] == "NOT IN LEDGER" for r in rows)))
    print("  ", paths["png"], "| copy in", FIGW)
apply_style()
d79 = pickle.load(open(R + "/full_variable/models/bn_final_model_iteration79.pkl", "rb")); E = sorted(tuple(e) for e in d79["model_edges"])
cells = json.load(open(R + "/selection/notebooks/02_structure.ipynb"))["cells"]
print("monitoring edges:", len(E), "| selection layout source: 02_structure cell index 5")

In [ ]:
# 11_paper_figures cell 2: Fig 2 evidence sensitivity.
def fig2():
    apply_style(); P = lambda f: pd.read_csv(R + "/" + f)
    MF, MFL, SF, SV = "full_variable/tables/rq2_sensitivity_table.csv", "full_variable/tables/rq2_ext_partA_table.csv", "selection/tables/sel_rq2_sensitivity_table.csv", "selection/tables/sel_voi_partA_table.csv"
    mon = P(MF); mon = mon[mon.measure == "S_exp"].set_index("variable"); fl = P(MFL).set_index("variable"); sel = P(SF).set_index("variable"); vo = P(SV); vo = vo[vo.r == 10].set_index("variable")
    assert np.isclose(vo.tau.iloc[0], 1 / 11); NM = lambda v: v.replace(" (bin)", "").replace("Credit Utilisation Rate", "Credit utilisation").replace("Days Past Due", "days past due").replace("Days Late", "days late").replace("Debt Ratio", "Debt ratio")
    mv = list(fl.index); sv = ["Age", "Debt Ratio"]; rows = []
    fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 4.3)); B = STYLE["BN"]
    spec = [(0, 0, mv, lambda v: (mon.loc[v, "mean"], mon.loc[v, "ci_low"], mon.loc[v, "ci_high"]), "Mean S$_{exp}$ (probability), monitoring", "%.4f", MF, lambda v: "variable=%s | measure=S_exp | mean" % v),
            (0, 1, mv, lambda v: tuple(100 * x for x in (fl.loc[v, "flip_rate"], fl.loc[v, "flip_ci_low"], fl.loc[v, "flip_ci_high"])), "Flip rate (%) at flag threshold 0.17, monitoring", "%.2f", MFL, lambda v: "variable=%s | flip_rate" % v),
            (1, 0, sv, lambda v: (sel.loc[v, "mean_S_exp"], sel.loc[v, "ci_lo"], sel.loc[v, "ci_hi"]), "Mean S$_{exp}$ (probability), selection", "%.5f", SF, lambda v: "variable=%s | mean_S_exp" % v),
            (1, 1, sv, lambda v: tuple(100 * x for x in (vo.loc[v, "flip_rate"], vo.loc[v, "flip_ci_lo"], vo.loc[v, "flip_ci_hi"])), "Flip rate (%) at flag threshold 1/11 (r = 10), selection", "%.1f", SV, lambda v: "variable=%s | r=10 | flip_rate" % v)]
    for k, (i, j, vs, f, xl, fm, src, loc) in enumerate(spec):
        ax = axes[i, j]; y = np.arange(len(vs))[::-1]; vals = [f(v) for v in vs]; xmax = max(c[2] for c in vals)
        for yy, v, (m, lo, hi) in zip(y, vs, vals):
            ax.errorbar([m], [yy], xerr=[[m - lo], [hi - m]], fmt=B["marker"], color=B["color"], ms=4, capsize=2, lw=1)
            ax.text(hi + 0.02 * xmax, yy, fm % m, va="center", ha="left", fontsize=7.5)
            rows.append(dict(label=("%s: %s" % (xl.split(",")[0], v)), display=fm % m, source_file=src, source_locator=loc(v), value=m / (100 if j == 1 else 1)))
        ax.set_yticks(y); ax.set_yticklabels([NM(v) for v in vs]); ax.set_xlim(0, xmax * 1.3); ax.set_ylim(-0.6, len(vs) - 0.4); ax.set_xlabel(xl)
        ax.text(-0.02, 1.04, "(%s)" % "abcd"[k], transform=ax.transAxes, fontweight="bold", ha="left", va="bottom")
    fig.text(0.01, 0.005, "Bars show the stored 95% intervals. All variables outside Default's Markov blanket have mean S$_{exp}$ exactly 0 in both stages (stored tables).\nFlip rate: share of test rows whose flag changes when the variable is retracted; selection flip rates depend on r = 10.", fontsize=7, va="bottom")
    fig.subplots_adjust(left=0.2, right=0.97, top=0.93, bottom=0.2, wspace=0.62, hspace=0.6)
    save_fig(fig, "sel_paper_fig2_sensitivity", map_ids(rows))
fig2()

In [ ]:
# 11_paper_figures cell 3: Fig 1 and Fig 3 (original style) + contact sheet.
from matplotlib.patches import Ellipse, Rectangle, RegularPolygon, FancyBboxPatch, FancyArrowPatch
import textwrap
LB, BL, GOLD, GREY_ = "#add8e6", "#7fb8dc", "gold", "lightgray"
def fig1_v2():
    apply_style()
    SE = [tuple(e) for e in json.load(open(R + "/selection/tables/sel_dag_9node_edges.json"))["edges"]]
    src = "".join(cells[5]["source"]) if isinstance(cells[5]["source"], list) else cells[5]["source"]
    i0 = src.index("pos = {'Default'"); k = src.index("{", i0); dpt = 0
    for j in range(k, len(src)):
        dpt += (src[j] == "{") - (src[j] == "}")
        if dpt == 0: break
    spos = ast.literal_eval(src[k:j + 1]); assert set(spos) >= {u for e in SE for u in e}
    HA = 3.0; HB = 2.25; HL = 0.4; HF = 0.3; HT = HA + HB + HL + HF
    XA = {"Credit Utilisation Rate (bin)": (1.0, 2.92), "90+ Days Late (bin)": (2.7, 2.0), "Open Credit Lines (bin)": (4.4, 2.45), "Default": (1.7, 1.5), "30-59 Days Past Due (bin)": (0.75, 0.45),
          "60-89 Days Past Due (bin)": (2.9, 0.5), "Debt Ratio (bin)": (5.45, 2.92), "Monthly Income (bin)": (5.45, 1.9), "Real Estate Loans (bin)": (4.5, 1.2), "Age (bin)": (6.0, 0.5),
          "Number of Dependents (bin)": (4.85, 0.4), "M_Income": (6.45, 1.9), "M_Dependents": (3.95, 0.4)}
    assert set(XA) == {u for e in E for u in e}
    xs = [p[0] for p in spos.values()]; ys = [p[1] for p in spos.values()]
    PB = {n: (0.62 + (x - min(xs)) / (max(xs) - min(xs)) * (WIDTH - 1.24), 0.4 + (y - min(ys)) / (max(ys) - min(ys)) * (HB - 0.8)) for n, (x, y) in spos.items()}
    NM = {"Credit Utilisation Rate": "Credit\nUtilisation Rate", "90+ Days Late": "90+ Days\nLate", "30-59 Days Past Due": "30-59 Days\nPast Due", "60-89 Days Past Due": "60-89 Days\nPast Due", "Open Credit Lines": "Open Credit\nLines",
          "Debt Ratio": "Debt\nRatio", "Monthly Income": "Monthly\nIncome", "Real Estate Loans": "Real Estate\nLoans", "Number of Dependents": "Number of\nDependents"}
    lab = lambda n: NM.get(n.replace(" (bin)", ""), n.replace(" (bin)", ""))
    IND = {"M_Income", "M_Dependents"}; EW, EH, RW, RH, HR = 1.1, 0.42, 0.62, 0.34, 0.3
    def blanket(ed): return {u for u, v in ed if v == "Default"} | {v for u, v in ed if u == "Default"}
    def size(n): return (RW, RH) if n in IND else ((2 * HR, 2 * HR) if n == "Default" else (EW, EH))
    def crossings(ed, pos):
        def ccw(a, b, c): return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
        n = 0
        for a in range(len(ed)):
            for b in range(a + 1, len(ed)):
                (p, q), (r, s) = ed[a], ed[b]
                if len({p, q, r, s}) < 4: continue
                A, B, C, D = pos[p], pos[q], pos[r], pos[s]; n += (ccw(A, C, D) != ccw(B, C, D)) and (ccw(A, B, C) != ccw(A, B, D))
        return n
    def overlaps(pos):
        ns = list(pos); c = 0
        for i in range(len(ns)):
            for j in range(i + 1, len(ns)):
                (w1, h1), (w2, h2) = size(ns[i]), size(ns[j])
                if abs(pos[ns[i]][0] - pos[ns[j]][0]) < (w1 + w2) / 2 and abs(pos[ns[i]][1] - pos[ns[j]][1]) < (h1 + h2) / 2: c += 1
        return c
    fig = plt.figure(figsize=(WIDTH, HT)); info = {}
    for tag, edges, pos, h0, hh, lo in (("a", E, XA, HB + HL + HF, HA, 0.15), ("b", SE, PB, HL + HF, HB, 0.0)):
        ax = fig.add_axes([0, h0 / HT, 1, hh / HT]); ax.set_xlim(0, WIDTH); ax.set_ylim(lo, lo + hh); ax.axis("off"); bl = blanket(edges); P = {}
        for n, (x, y) in pos.items():
            if n == "Default": p = RegularPolygon((x, y), 6, radius=HR, orientation=np.pi / 6, fc=GOLD, ec="black", lw=1.6)
            elif n in IND: p = Rectangle((x - RW / 2, y - RH / 2), RW, RH, fc=GREY_, ec="black", lw=0.8)
            elif n in bl: p = FancyBboxPatch((x - EW / 2, y - EH / 2), EW, EH, boxstyle="round,pad=0,rounding_size=0.09", fc=BL, ec="black", lw=1.4)
            else: p = Ellipse((x, y), EW, EH, fc=LB, ec="black", lw=0.8)
            ax.add_patch(p); P[n] = p; ax.text(x, y, lab(n), ha="center", va="center", fontsize=7, zorder=5)
        for u, v in edges:
            ind = u in IND
            ax.add_patch(FancyArrowPatch(pos[u], pos[v], patchA=P[u], patchB=P[v], arrowstyle="-|>", mutation_scale=7, connectionstyle="arc3,rad=0.08", color="dimgray" if ind else "0.3", lw=1.2 if ind else 0.75, ls="--" if ind else "-", shrinkA=0, shrinkB=0, zorder=2))
        ax.text(0.01, 0.98, "(%s)" % tag, transform=ax.transAxes, fontweight="bold", fontsize=8, va="top"); info[tag] = (len(pos), len(edges), crossings(edges, pos), overlaps(pos))
    la = fig.add_axes([0, HF / HT, 1, HL / HT]); la.set_xlim(0, WIDTH); la.set_ylim(0, HL); la.axis("off")
    for (x, y, kind, t) in ((0.25, 0.27, "h", "Default (target)"), (0.25, 0.09, "b", "Markov blanket of Default"), (3.2, 0.27, "e", "other variable"), (3.2, 0.09, "s", "missingness indicator (dashed edges)")):
        if kind == "h": la.add_patch(RegularPolygon((x, y), 6, radius=0.075, orientation=np.pi / 6, fc=GOLD, ec="black", lw=1.2))
        elif kind == "b": la.add_patch(FancyBboxPatch((x - 0.11, y - 0.06), 0.22, 0.12, boxstyle="round,pad=0,rounding_size=0.03", fc=BL, ec="black", lw=1.0))
        elif kind == "e": la.add_patch(Ellipse((x, y), 0.22, 0.12, fc=LB, ec="black", lw=0.8))
        else: la.add_patch(Rectangle((x - 0.08, y - 0.06), 0.16, 0.12, fc=GREY_, ec="black", lw=0.8))
        la.text(x + 0.22, y, t, fontsize=7.5, va="center")
    fig.text(0.01, 0.01, "\n".join(textwrap.wrap("Edges point from parent to child; parents are drawn above Default and children below. (a) monitoring BN, 13 nodes, hand-placed layout (the original Graphviz layout was not saved); (b) selection BN, 9 nodes, original hand-placed layout.", 118)), fontsize=7, va="bottom")
    print("Fig 1 (nodes, edges, straight-edge crossings, node-box overlaps):", info, "| height %.2f in (budget 6.0)" % HT)
    save_fig(fig, "sel_paper_fig1_structures", [])
fig1_v2()
def contact():
    from PIL import ImageDraw
    fs = [("original: bn_13node_dag_final", R + "/full_variable/figures/bn_13node_dag_final.png"), ("original: sel_dag_9node_v3", R + "/selection/figures/sel_dag_9node_v3.png"), ("new Fig 1", R + "/selection/figures/sel_paper_fig1_structures.png")]
    ims = []
    for t, p in fs:
        im = Image.open(p).convert("RGB"); w = 1000; im = im.resize((w, int(im.height * w / im.width))); ims.append((t, im))
    H = max(i.height for _, i in ims) + 40; sheet = Image.new("RGB", (3 * 1000 + 40, H), "white"); d = ImageDraw.Draw(sheet)
    for k, (t, im) in enumerate(ims): sheet.paste(im, (k * 1010, 36)); d.text((k * 1010 + 4, 8), t + " (equal widths here; new Fig 1 is 6.83 in wide at 100%)", fill="black")
    sheet.save(FIGW + "fig1_contact_sheet.png"); print("contact sheet:", FIGW + "fig1_contact_sheet.png")
contact()
def fig3_v2():
    apply_style(); from matplotlib.lines import Line2D
    CF, DV = "selection/tables/sel_default_cells.csv", "selection/tables/sel_decision_value_combined.csv"
    d = pd.read_csv(R + "/" + CF); ages = list(dict.fromkeys(d.age_bin)); drs = list(dict.fromkeys(d.debt_ratio_bin))
    M = d.pivot(index="age_bin", columns="debt_ratio_bin", values="p_default_em").loc[ages, drs]
    dv = pd.read_csv(R + "/" + DV); tau = dv[(dv.contribution == "selection") & (dv.model == "BN")].set_index("r").tau; t10, t5 = float(tau[10]), float(tau[5]); assert np.isclose(t10, 1 / 11) and np.isclose(t5, 1 / 6)
    s4 = json.load(open(R + "/selection/notebooks/04_baselines_rq2.ipynb"))["cells"][14]; s4 = "".join(s4["source"]) if isinstance(s4["source"], list) else s4["source"]
    m = [l for l in s4.splitlines() if "imshow(" in l]; mm = re.search(r"cmap\s*=\s*[\"']([A-Za-z_]+)[\"']", " ".join(m)) if m else None; orig = mm.group(1) if mm else None
    def mono(name):
        try:
            c = mpl.colormaps[name]; L = np.array([(0.299, 0.587, 0.114) @ np.array(c(x)[:3]) for x in np.linspace(0, 1, 64)])
        except Exception: return False
        dd = np.diff(L); return bool((dd >= -1e-3).all() or (dd <= 1e-3).all())
    cmap = orig if (orig and mono(orig)) else "YlOrRd"; print("original heatmap colormap:", orig, "| used:", cmap)
    cm = mpl.colormaps[cmap]; vmax = t5 * 1.03; fr = lambda t: "1/%d" % round(1 / t); nice = lambda s: s.replace("-inf", "\u2212\u221e").replace("inf", "\u221e")
    HT = 4.3; fig = plt.figure(figsize=(WIDTH, HT)); ax = fig.add_axes([0.13, 0.40, 0.57, 0.57]); cax = fig.add_axes([0.84, 0.40, 0.025, 0.52])
    im = ax.imshow(M.values, cmap=cm, vmin=0, vmax=vmax, aspect="auto"); rows = []
    for i, a in enumerate(ages):
        for j, b in enumerate(drs):
            v = M.iloc[i, j]; rgb = cm(v / vmax)[:3]; lum = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
            ax.text(j, i, "%.4f" % v, ha="center", va="center", fontsize=8, color="black" if lum > 0.5 else "white")
            if v >= t10: ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=2.4)); ax.add_patch(Rectangle((j - 0.46, i - 0.46), 0.92, 0.92, fill=False, ec="white", lw=1.0, ls=":"))
            rows.append(dict(label="Default probability, Age %s x Debt ratio %s" % (a, b), display="%.4f" % v, source_file=CF, source_locator="cell age_bin=%s | debt_ratio_bin=%s | p_default_em" % (a, b), value=v))
    n_hi = int((M.values >= t10).sum()); assert int((M.values >= t5).sum()) == 0
    ax.set_xticks(range(5)); ax.set_xticklabels([nice(x) for x in drs], rotation=15); ax.set_yticks(range(5)); ax.set_yticklabels([nice(x) for x in ages]); ax.set_xlabel("Debt ratio bin"); ax.set_ylabel("Age bin")
    [s.set_visible(True) for s in ax.spines.values()]
    cb = fig.colorbar(im, cax=cax); cb.ax.text(0.5, 1.04, "P(Default = 1)", transform=cb.ax.transAxes, ha="center", va="bottom", fontsize=8)
    for t, r in ((t10, 10), (t5, 5)): cb.ax.axhline(t, color="black", lw=1.4); cb.ax.text(-0.35, t / vmax, "%s (r = %d) \u2192" % (fr(t), r), va="center", ha="right", fontsize=7.5, transform=cb.ax.transAxes)
    fig.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor="none", markeredgecolor="black", markeredgewidth=2, ms=9, label="outlined cell: at or above the flag threshold %s (r = 10); %d of 25 cells" % (fr(t10), n_hi)),
                        Line2D([0], [0], color="w", label="no cell reaches the flag threshold %s (r = 5), so the BN flags nobody at r = 5" % fr(t5))], loc="lower left", bbox_to_anchor=(0.02, 0.07), frameon=False, fontsize=8, ncol=1)
    fig.text(0.01, 0.008, "Cell values are training default rates (EM fit equals plain counts on the fitted rows).", fontsize=7, va="bottom")
    ng = append_g10(map_ids(rows), "selection", lambda r: r["label"]); print("G10 ledger rows appended this run:", ng, "| ledger rows now:", len(load_led()))
    save_fig(fig, "sel_paper_fig3_lookup_heatmap", rows)
fig3_v2()

In [ ]:
# 11_paper_figures cell 4: READ-ONLY inspection for Fig 1 iteration 3. Writes one image to WRITEUP/figures.
import scipy.ndimage as ndi
from PIL import ImageDraw
def inspect_fig1():
    cs = cells[5]; s = "".join(cs["source"]) if isinstance(cs["source"], list) else cs["source"]
    print("=== original drawing code, 02_structure cell index 5 (first 90 lines) ===")
    print("\n".join(s.splitlines()[:90]))
    p = R + "/full_variable/figures/bn_13node_dag_final.png"; im = Image.open(p).convert("RGB"); f = 4
    sm = im.resize((im.width // f, im.height // f), Image.NEAREST); a = np.asarray(sm).astype(int)
    def mask(rgb, tol): return (np.abs(a - np.array(rgb)).max(axis=2) <= tol)
    classes = {"gold": ((255, 215, 0), 8), "grey square": ((211, 211, 211), 4), "light blue": ((173, 216, 230), 6)}
    ov = sm.copy(); dr = ImageDraw.Draw(ov); k = 0; found = []
    for cname, (rgb, tol) in classes.items():
        m = ndi.binary_fill_holes(ndi.binary_closing(mask(rgb, tol), iterations=2)); lab, n = ndi.label(m)
        for i in range(1, n + 1):
            ys, xs = np.where(lab == i)
            if len(xs) < 400: continue
            k += 1; cx, cy = xs.mean() * f, ys.mean() * f; found.append((k, cname, cx, cy, len(xs) * f * f, (xs.max() - xs.min()) * f, (ys.max() - ys.min()) * f))
            dr.text((xs.mean() - 8, ys.mean() - 8), str(k), fill="red")
    print("\n=== nodes detected in bn_13node_dag_final.png (pixel coordinates, image %d x %d) ===" % im.size)
    for r in found: print("  #%-2d %-12s centre (%.0f, %.0f) | width %d height %d px" % (r[0], r[1], r[2], r[3], r[5], r[6]))
    print("expected 13 nodes: 1 gold, 2 grey squares, 10 light blue | found:", len(found))
    ov.save(FIGW + "fig1_original_a_numbered.png"); print("numbered overlay:", FIGW + "fig1_original_a_numbered.png")
inspect_fig1()

In [ ]:
# 11_paper_figures cell 5: Fig 1 iteration 3 (final). Originals' layout and style; labels bold 9 pt as in the originals.
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
def fig1_final():
    apply_style(); FONT = "DejaVu Sans"
    SE = [tuple(e) for e in json.load(open(R + "/selection/tables/sel_dag_9node_edges.json"))["edges"]]
    PA = {"Default": (6092, 1588), "M_Income": (3986, 2082), "M_Dependents": (4146, 3574), "Credit Utilisation Rate (bin)": (3726, 594), "90+ Days Late (bin)": (4947, 1091),
          "Open Credit Lines (bin)": (2422, 1588), "Debt Ratio (bin)": (645, 2084), "60-89 Days Past Due (bin)": (5158, 2084), "Monthly Income (bin)": (3043, 2581),
          "30-59 Days Past Due (bin)": (6520, 2581), "Real Estate Loans (bin)": (1751, 3078), "Age (bin)": (1751, 3574), "Number of Dependents (bin)": (3043, 4071)}
    PBU = {'Default': (0, 0), 'Age (bin)': (-2, 1.3), 'Debt Ratio (bin)': (2, 1.3), 'Monthly Income (bin)': (4.2, 3.2), 'Open Credit Lines (bin)': (0, 3.2),
           'Real Estate Loans (bin)': (-4.2, 3.2), 'Number of Dependents (bin)': (-4.2, 5.2), 'M_Income': (4.2, 5.2), 'M_Dependents': (-6.4, 5.2)}
    assert set(PA) == {u for e in E for u in e} and set(PBU) == {u for e in SE for u in e}
    HL, HA, HB = 0.3, 3.3, 2.4; HT = HL + HA + HB; XL, XR = 0.45, 5.0; RAD = 0.15
    def scale(P, h, flip):
        xs = [p[0] for p in P.values()]; ys = [p[1] for p in P.values()]; out = {}
        for n, (x, y) in P.items():
            fx = (x - min(xs)) / (max(xs) - min(xs)); fy = (y - min(ys)) / (max(ys) - min(ys))
            out[n] = (XL + fx * (XR - XL), 0.2 + ((1 - fy) if flip else fy) * (h - 0.4))
        return out
    A = scale(PA, HA, True); B = scale(PBU, HB, False)
    NM = {"Credit Utilisation Rate": "Credit\nUtilisation\nRate", "90+ Days Late": "90+ Days\nLate", "30-59 Days Past Due": "30-59 Days\nPast Due", "60-89 Days Past Due": "60-89 Days\nPast Due",
          "Open Credit Lines": "Open Credit\nLines", "Debt Ratio": "Debt Ratio", "Monthly Income": "Monthly\nIncome", "Real Estate Loans": "Real Estate\nLoans", "Number of Dependents": "Number of\nDependents"}
    lab = lambda n: NM.get(n.replace(" (bin)", ""), n.replace(" (bin)", ""))
    IND = {"M_Income", "M_Dependents"}
    def crossings(ed, pos):
        def ccw(a, b, c): return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
        n = 0
        for i in range(len(ed)):
            for j in range(i + 1, len(ed)):
                (p, q), (r, s) = ed[i], ed[j]
                if len({p, q, r, s}) < 4: continue
                a, b, c, d = pos[p], pos[q], pos[r], pos[s]; n += (ccw(a, c, d) != ccw(b, c, d)) and (ccw(a, b, c) != ccw(a, b, d))
        return n
    fig = plt.figure(figsize=(WIDTH, HT)); info = {}
    for tag, edges, pos, h0, hh in (("a", E, A, HL + HB, HA), ("b", SE, B, HL, HB)):
        ax = fig.add_axes([0, h0 / HT, 1, hh / HT]); ax.set_xlim(0, WIDTH); ax.set_ylim(0, hh); ax.axis("off")
        bl = {u for u, v in edges if v == "Default"} | {v for u, v in edges if u == "Default"}; P = {}
        for n, (x, y) in pos.items():
            if n == "Default": p = Circle((x, y), RAD, fc="gold", ec="black", lw=1.2)
            elif n in IND: p = Rectangle((x - RAD, y - RAD), 2 * RAD, 2 * RAD, fc="lightgray", ec="black", lw=0.8)
            elif n in bl: p = Circle((x, y), RAD, fc="lightblue", ec="darkorange", lw=2.2)
            else: p = Circle((x, y), RAD, fc="lightblue", ec="black", lw=0.8)
            ax.add_patch(p); P[n] = p
            t = lab(n); wd = max(len(l) for l in t.split("\n")) * 0.092; left = x + RAD + 0.05 + wd > WIDTH - 0.03
            ax.text(x - RAD - 0.04 if left else x + RAD + 0.04, y, t, ha="right" if left else "left", va="center", fontsize=9, fontweight="bold", family=FONT, zorder=6,
                    bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.4))
        for u, v in edges:
            ind = u in IND
            ax.add_patch(FancyArrowPatch(pos[u], pos[v], patchA=P[u], patchB=P[v], arrowstyle="-|>", mutation_scale=6, connectionstyle="arc3,rad=0.08",
                                         color="dimgray" if ind else "gray", lw=1.1 if ind else 0.6, ls="--" if ind else "-", shrinkA=0, shrinkB=0, zorder=2))
        ax.text(0.005, 0.99, "(%s)" % tag, transform=ax.transAxes, fontweight="bold", fontsize=9, va="top", family=FONT); info[tag] = (len(pos), len(edges), crossings(edges, pos))
    la = fig.add_axes([0, 0, 1, HL / HT]); la.set_xlim(0, WIDTH); la.set_ylim(0, HL); la.axis("off"); y = HL / 2
    for x, kind, t in ((0.2, "d", "Default"), (1.0, "b", "Markov blanket of Default"), (3.0, "o", "other variable"), (4.3, "s", "missingness indicator (dashed edges)")):
        if kind == "d": la.add_patch(Circle((x, y), 0.07, fc="gold", ec="black", lw=1.0))
        elif kind == "b": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="darkorange", lw=1.6))
        elif kind == "o": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="black", lw=0.8))
        else: la.add_patch(Rectangle((x - 0.07, y - 0.07), 0.14, 0.14, fc="lightgray", ec="black", lw=0.8))
        la.text(x + 0.13, y, t, fontsize=7.5, va="center", family=FONT)
    print("Fig 1 (nodes, edges, straight-edge crossings):", info, "| total height %.2f in (budget 6.0)" % HT)
    save_fig(fig, "sel_paper_fig1_structures", [])
fig1_final()
def contact2():
    from PIL import ImageDraw
    fs = [("original (a) bn_13node_dag_final", R + "/full_variable/figures/bn_13node_dag_final.png"), ("original (b) sel_dag_9node_v3", R + "/selection/figures/sel_dag_9node_v3.png"), ("new Fig 1", R + "/selection/figures/sel_paper_fig1_structures.png")]
    ims = [(t, Image.open(p).convert("RGB")) for t, p in fs]; ims = [(t, i.resize((1000, int(i.height * 1000 / i.width)))) for t, i in ims]
    sheet = Image.new("RGB", (3030, max(i.height for _, i in ims) + 40), "white"); d = ImageDraw.Draw(sheet)
    for k, (t, i) in enumerate(ims): sheet.paste(i, (k * 1010, 36)); d.text((k * 1010 + 4, 8), t + " (equal width)", fill="black")
    sheet.save(FIGW + "fig1_contact_sheet.png"); print("contact sheet:", FIGW + "fig1_contact_sheet.png")
contact2()

In [ ]:
# 11_paper_figures cell 5: Fig 1 final pass. (a) positions read off bn_13node_dag_final.png; (b) read off sel_dag_9node_v3.png.
import scipy.ndimage as ndi, matplotlib.patheffects as pe
from PIL import ImageDraw
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
def detect(path, f=4):
    im = Image.open(path).convert("RGB"); sm = im.resize((im.width // f, im.height // f), Image.NEAREST); a = np.asarray(sm).astype(int)
    org = np.abs(a - np.array([255, 140, 0])).max(axis=2) <= 40; res = {"gold": [], "grey": [], "blue": []}
    for c, rgb, tol in (("gold", (255, 215, 0), 8), ("grey", (211, 211, 211), 4), ("blue", (173, 216, 230), 6)):
        m = ndi.binary_fill_holes(ndi.binary_closing(np.abs(a - np.array(rgb)).max(axis=2) <= tol, iterations=2)); lab, n = ndi.label(m)
        for i in range(1, n + 1):
            ys, xs = np.where(lab == i)
            if len(xs) < 400: continue
            o = int(org[max(0, ys.min() - 8):ys.max() + 8, max(0, xs.min() - 8):xs.max() + 8].sum())
            res[c].append((xs.mean() * f, ys.mean() * f, o > 150))
    return res, im
def crossings(ed, pos):
    def ccw(a, b, c): return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
    n = 0
    for i in range(len(ed)):
        for j in range(i + 1, len(ed)):
            (p, q), (r, s) = ed[i], ed[j]
            if len({p, q, r, s}) < 4: continue
            a, b, c, d = pos[p], pos[q], pos[r], pos[s]; n += (ccw(a, c, d) != ccw(b, c, d)) and (ccw(a, b, c) != ccw(a, b, d))
    return n
def fig1_final2():
    apply_style(); FONT = "DejaVu Sans"
    SE = [tuple(e) for e in json.load(open(R + "/selection/tables/sel_dag_9node_edges.json"))["edges"]]
    PA = {"Default": (6092, 1588), "M_Income": (3986, 2082), "M_Dependents": (4146, 3574), "Credit Utilisation Rate (bin)": (3726, 594), "90+ Days Late (bin)": (4947, 1091),
          "Open Credit Lines (bin)": (2422, 1588), "Debt Ratio (bin)": (645, 2084), "60-89 Days Past Due (bin)": (5158, 2084), "Monthly Income (bin)": (3043, 2581),
          "30-59 Days Past Due (bin)": (6520, 2581), "Real Estate Loans (bin)": (1751, 3078), "Age (bin)": (1751, 3574), "Number of Dependents (bin)": (3043, 4071)}
    det, imb = detect(R + "/selection/figures/sel_dag_9node_v3.png")
    assert len(det["gold"]) == 1 and len(det["grey"]) == 2 and len(det["blue"]) == 6, "panel (b) detection: %s" % {k: len(v) for k, v in det.items()}
    dist = lambda p, q: np.hypot(p[0] - q[0], p[1] - q[1]); g = sorted(det["grey"], key=lambda b: b[1]); MIc, MDp = g[0][:2], g[1][:2]
    orange = sorted([b for b in det["blue"] if b[2]], key=lambda b: b[1]); blue = [b for b in det["blue"] if not b[2]]
    assert len(orange) == 2 and len(blue) == 4, "orange/blue split: %d/%d" % (len(orange), len(blue))
    Age, Debt = orange[1][:2], orange[0][:2]; MonInc = min(blue, key=lambda b: dist(b, MIc)); rest = [b for b in blue if b is not MonInc]; NumDep = min(rest, key=lambda b: dist(b, MDp)); pair = [b for b in rest if b is not NumDep]
    base = {"Default": det["gold"][0][:2], "M_Income": MIc, "M_Dependents": MDp, "Age (bin)": Age, "Debt Ratio (bin)": Debt, "Monthly Income (bin)": MonInc[:2], "Number of Dependents (bin)": NumDep[:2]}
    cand = []
    for k in (0, 1):
        P = dict(base); P["Real Estate Loans (bin)"] = pair[k][:2]; P["Open Credit Lines (bin)"] = pair[1 - k][:2]; cand.append((crossings(SE, P), P))
    print("panel (b): crossings for the two Real Estate Loans / Open Credit Lines assignments:", [c for c, _ in cand], "(tie: first kept)" if cand[0][0] == cand[1][0] else "")
    PBU = min(cand, key=lambda c: c[0])[1]; assert set(PBU) == {u for e in SE for u in e}
    ov = imb.copy(); dr = ImageDraw.Draw(ov)
    for n, (x, y) in PBU.items(): dr.text((x - 60, y), n.replace(" (bin)", ""), fill="red")
    ov.resize((1400, int(ov.height * 1400 / ov.width))).save(FIGW + "fig1_original_b_assigned.png")
    HL, HA, HB = 0.3, 3.3, 2.4; HT = HL + HA + HB; RAD = 0.15
    def scale(P, h, xl, xr):
        xs = [p[0] for p in P.values()]; ys = [p[1] for p in P.values()]
        return {n: (xl + (x - min(xs)) / (max(xs) - min(xs)) * (xr - xl), 0.35 + (1 - (y - min(ys)) / (max(ys) - min(ys))) * (h - 0.7)) for n, (x, y) in P.items()}
    A = scale(PA, HA, 0.45, 5.0); B = scale(PBU, HB, 0.5, 5.4)
    NM = {"Credit Utilisation Rate": "Credit\nUtilisation\nRate", "90+ Days Late": "90+ Days\nLate", "30-59 Days Past Due": "30-59 Days\nPast Due", "60-89 Days Past Due": "60-89 Days\nPast Due",
          "Open Credit Lines": "Open Credit\nLines", "Debt Ratio": "Debt Ratio", "Monthly Income": "Monthly\nIncome", "Real Estate Loans": "Real Estate\nLoans", "Number of Dependents": "Number of\nDependents"}
    lab = lambda n: NM.get(n.replace(" (bin)", ""), n.replace(" (bin)", "")); IND = {"M_Income", "M_Dependents"}
    def place(pos, edges, hh):
        circ = list(pos.values()); boxes = [(0.0, hh - 0.28, 0.32, hh)]; pts = []
        for u, v in edges:
            for t in np.linspace(0, 1, 40):
                p = (pos[u][0] * (1 - t) + pos[v][0] * t, pos[u][1] * (1 - t) + pos[v][1] * t)
                if all(np.hypot(p[0] - c[0], p[1] - c[1]) > RAD for c in circ): pts.append(p)
        deg = {n: sum(n in e for e in edges) for n in pos}; out = {}; resid = 0
        for n in sorted(pos, key=lambda n: -deg[n]):
            x, y = pos[n]; t = lab(n); w = max(len(l) for l in t.split("\n")) * 0.1; h = (t.count("\n") + 1) * 0.16; best = None
            for side in ("r", "l", "a", "b"):
                if side == "r": b = (x + RAD + 0.04, y - h / 2, x + RAD + 0.04 + w, y + h / 2); an = (b[0], y, "left", "center")
                elif side == "l": b = (x - RAD - 0.04 - w, y - h / 2, x - RAD - 0.04, y + h / 2); an = (b[2], y, "right", "center")
                elif side == "a": b = (x - w / 2, y + RAD + 0.03, x + w / 2, y + RAD + 0.03 + h); an = (x, b[1], "center", "bottom")
                else: b = (x - w / 2, y - RAD - 0.03 - h, x + w / 2, y - RAD - 0.03); an = (x, b[3], "center", "top")
                s = 0
                if b[0] < 0.02 or b[2] > WIDTH - 0.02 or b[1] < 0.02 or b[3] > hh - 0.02: s += 1000
                for c in circ:
                    nx, ny = min(max(c[0], b[0]), b[2]), min(max(c[1], b[1]), b[3])
                    if np.hypot(nx - c[0], ny - c[1]) < RAD: s += 100
                for o in boxes:
                    if b[0] < o[2] and b[2] > o[0] and b[1] < o[3] and b[3] > o[1]: s += 100
                hits = sum(b[0] <= p[0] <= b[2] and b[1] <= p[1] <= b[3] for p in pts); s += hits
                if best is None or s < best[0]: best = (s, b, an, hits)
            boxes.append(best[1]); out[n] = (t, best[2]); resid += best[0]
        return out, resid
    fig = plt.figure(figsize=(WIDTH, HT)); info = {}
    for tag, edges, pos, h0, hh in (("a", E, A, HL + HB, HA), ("b", SE, B, HL, HB)):
        ax = fig.add_axes([0, h0 / HT, 1, hh / HT]); ax.set_xlim(0, WIDTH); ax.set_ylim(0, hh); ax.axis("off")
        bl = {u for u, v in edges if v == "Default"} | {v for u, v in edges if u == "Default"}; P = {}
        for n, (x, y) in pos.items():
            if n == "Default": p = Circle((x, y), RAD, fc="gold", ec="black", lw=1.2)
            elif n in IND: p = Rectangle((x - RAD, y - RAD), 2 * RAD, 2 * RAD, fc="lightgray", ec="black", lw=0.8)
            elif n in bl: p = Circle((x, y), RAD, fc="lightblue", ec="darkorange", lw=2.2)
            else: p = Circle((x, y), RAD, fc="lightblue", ec="black", lw=0.8)
            ax.add_patch(p); P[n] = p
        drawn = set()
        for u, v in edges:
            ind = u in IND; drawn.add((u, v))
            ax.add_patch(FancyArrowPatch(pos[u], pos[v], patchA=P[u], patchB=P[v], arrowstyle="-|>", mutation_scale=6, connectionstyle="arc3,rad=0.08", color="dimgray" if ind else "gray", lw=1.1 if ind else 0.6, ls="--" if ind else "-", shrinkA=0, shrinkB=0, zorder=2))
        assert drawn == set(edges) and len(drawn) == len(edges), "drawn edge set differs from stored structure"
        lp, resid = place(pos, edges, hh)
        for n, (t, (ax_, ay_, ha, va)) in lp.items():
            ax.text(ax_, ay_, t, ha=ha, va=va, fontsize=9, fontweight="bold", family=FONT, zorder=6, linespacing=1.0, path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])
        ax.text(0.005, 0.99, "(%s)" % tag, transform=ax.transAxes, fontweight="bold", fontsize=9, va="top", family=FONT); info[tag] = (len(pos), len(edges), crossings(edges, pos), "residual label score %d" % resid)
    la = fig.add_axes([0, 0, 1, HL / HT]); la.set_xlim(0, WIDTH); la.set_ylim(0, HL); la.axis("off"); y = HL / 2
    for x, kind, t in ((0.2, "d", "Default"), (1.0, "b", "Markov blanket of Default"), (3.0, "o", "other variable"), (4.3, "s", "missingness indicator (dashed edges)")):
        if kind == "d": la.add_patch(Circle((x, y), 0.07, fc="gold", ec="black", lw=1.0))
        elif kind == "b": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="darkorange", lw=1.6))
        elif kind == "o": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="black", lw=0.8))
        else: la.add_patch(Rectangle((x - 0.07, y - 0.07), 0.14, 0.14, fc="lightgray", ec="black", lw=0.8))
        la.text(x + 0.13, y, t, fontsize=7.5, va="center", family=FONT)
    print("Fig 1 (nodes, edges, straight-edge crossings, label score; 0 = no label hits):", info, "| height %.2f in" % HT)
    save_fig(fig, "sel_paper_fig1_structures", [])
fig1_final2()
fs = [("original (a)", R + "/full_variable/figures/bn_13node_dag_final.png"), ("original (b)", R + "/selection/figures/sel_dag_9node_v3.png"), ("new Fig 1", R + "/selection/figures/sel_paper_fig1_structures.png")]
ims = [(t, Image.open(p).convert("RGB")) for t, p in fs]; ims = [(t, i.resize((1000, int(i.height * 1000 / i.width)))) for t, i in ims]
sheet = Image.new("RGB", (3030, max(i.height for _, i in ims) + 40), "white"); d = ImageDraw.Draw(sheet)
for k, (t, i) in enumerate(ims): sheet.paste(i, (k * 1010, 36)); d.text((k * 1010 + 4, 8), t + " (equal width)", fill="black")
sheet.save(FIGW + "fig1_contact_sheet.png"); print("contact sheet saved")

In [ ]:
# 11_paper_figures cell 5b: Fig 1, labels INSIDE circles. Renames the current (beside) files to *_beside first.
import scipy.ndimage as ndi
from PIL import ImageDraw
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
from matplotlib import font_manager as fm
def check_style(fig):   # redefined: 6 pt floor for node labels (gid "node"), 7 pt for everything else
    assert abs(fig.get_figwidth() - WIDTH) < 1e-6, "figure width is not 6.83 in"
    assert all(ax.get_title() == "" for ax in fig.axes) and fig._suptitle is None, "title inside figure"
    ts = [t for t in fig.findobj(mtext.Text) if t.get_visible() and t.get_text().strip()]
    bad = [(t.get_text()[:30], t.get_fontsize()) for t in ts if t.get_fontsize() < (6 if t.get_gid() == "node" else 7) - 1e-9]; assert not bad, "text under floor: %s" % bad[:5]
    w = [t.get_text() for t in ts if BANNED.search(t.get_text())]; assert not w, "banned wording: %s" % w
    return min(t.get_fontsize() for t in ts)
for e in ("png", "pdf"):
    for base in (R + "/selection/figures/", FIGW):
        p = base + "sel_paper_fig1_structures." + e
        if os.path.exists(p): assert not os.path.exists(base + "sel_paper_fig1_structures_beside." + e); os.rename(p, base + "sel_paper_fig1_structures_beside." + e)
if os.path.exists(FIGW + "sel_paper_fig1_structures_numbers.csv"): os.rename(FIGW + "sel_paper_fig1_structures_numbers.csv", FIGW + "sel_paper_fig1_structures_beside_numbers.csv")
def detect(path, f=4):
    im = Image.open(path).convert("RGB"); sm = im.resize((im.width // f, im.height // f), Image.NEAREST); a = np.asarray(sm).astype(int)
    org = np.abs(a - np.array([255, 140, 0])).max(axis=2) <= 40; res = {"gold": [], "grey": [], "blue": []}
    for c, rgb, tol in (("gold", (255, 215, 0), 8), ("grey", (211, 211, 211), 4), ("blue", (173, 216, 230), 6)):
        m = ndi.binary_fill_holes(ndi.binary_closing(np.abs(a - np.array(rgb)).max(axis=2) <= tol, iterations=2)); lab, n = ndi.label(m)
        for i in range(1, n + 1):
            ys, xs = np.where(lab == i)
            if len(xs) < 400: continue
            o = int(org[max(0, ys.min() - 8):ys.max() + 8, max(0, xs.min() - 8):xs.max() + 8].sum()); res[c].append((xs.mean() * f, ys.mean() * f, o > 150))
    return res
def crossings(ed, pos):
    def ccw(a, b, c): return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])
    n = 0
    for i in range(len(ed)):
        for j in range(i + 1, len(ed)):
            (p, q), (r, s) = ed[i], ed[j]
            if len({p, q, r, s}) < 4: continue
            a, b, c, d = pos[p], pos[q], pos[r], pos[s]; n += (ccw(a, c, d) != ccw(b, c, d)) and (ccw(a, b, c) != ccw(a, b, d))
    return n
def fig1_inside():
    apply_style()
    names = {f.name for f in fm.fontManager.ttflist}; FONT = next((f for f in ("Liberation Sans", "Arial", "Helvetica") if f in names), "DejaVu Sans"); print("font used:", FONT)
    SE = [tuple(e) for e in json.load(open(R + "/selection/tables/sel_dag_9node_edges.json"))["edges"]]
    PA = {"Default": (6092, 1588), "M_Income": (3986, 2082), "M_Dependents": (4146, 3574), "Credit Utilisation Rate (bin)": (3726, 594), "90+ Days Late (bin)": (4947, 1091),
          "Open Credit Lines (bin)": (2422, 1588), "Debt Ratio (bin)": (645, 2084), "60-89 Days Past Due (bin)": (5158, 2084), "Monthly Income (bin)": (3043, 2581),
          "30-59 Days Past Due (bin)": (6520, 2581), "Real Estate Loans (bin)": (1751, 3078), "Age (bin)": (1751, 3574), "Number of Dependents (bin)": (3043, 4071)}
    det = detect(R + "/selection/figures/sel_dag_9node_v3.png")
    assert len(det["gold"]) == 1 and len(det["grey"]) == 2 and len(det["blue"]) == 6
    dist = lambda p, q: np.hypot(p[0] - q[0], p[1] - q[1]); g = sorted(det["grey"], key=lambda b: b[1]); MIc, MDp = g[0][:2], g[1][:2]
    orange = sorted([b for b in det["blue"] if b[2]], key=lambda b: b[1]); blue = [b for b in det["blue"] if not b[2]]; assert len(orange) == 2 and len(blue) == 4
    Age, Debt = orange[1][:2], orange[0][:2]; MonInc = min(blue, key=lambda b: dist(b, MIc)); rest = [b for b in blue if b is not MonInc]; NumDep = min(rest, key=lambda b: dist(b, MDp)); pair = [b for b in rest if b is not NumDep]
    base = {"Default": det["gold"][0][:2], "M_Income": MIc, "M_Dependents": MDp, "Age (bin)": Age, "Debt Ratio (bin)": Debt, "Monthly Income (bin)": MonInc[:2], "Number of Dependents (bin)": NumDep[:2]}
    cand = []
    for k in (0, 1):
        P = dict(base); P["Real Estate Loans (bin)"] = pair[k][:2]; P["Open Credit Lines (bin)"] = pair[1 - k][:2]; cand.append((crossings(SE, P), P))
    PB0 = min(cand, key=lambda c: c[0])[1]; assert set(PB0) == {u for e in SE for u in e}
    RAD = 0.33; DMIN = 0.70; MX = RAD + 0.08; MY = RAD + 0.12; HL = 0.3; BUDGET = 7.5
    def scale(P):
        xs = [p[0] for p in P.values()]; ys = [p[1] for p in P.values()]; sx = (WIDTH - 2 * MX) / (max(xs) - min(xs)); sy = 0.0; ns = list(P)
        for i in range(len(ns)):
            for j in range(i + 1, len(ns)):
                dx = abs(P[ns[i]][0] - P[ns[j]][0]) * sx; dy = abs(P[ns[i]][1] - P[ns[j]][1])
                if dx < DMIN:
                    assert dy > 0, "two nodes at the same height and closer than DMIN: %s %s" % (ns[i], ns[j])
                    sy = max(sy, np.sqrt(DMIN ** 2 - dx ** 2) / dy)
        sy = max(sy, 0.0005); h = (max(ys) - min(ys)) * sy + 2 * MY
        return {n: (MX + (x - min(xs)) * sx, MY + (max(ys) - y) * sy) for n, (x, y) in P.items()}, h
    A, hA = scale(PA); B, hB = scale(PB0); HT = HL + hA + hB
    print("panel heights: (a) %.2f in, (b) %.2f in, legend %.2f in | total %.2f in (budget %.1f)%s" % (hA, hB, HL, HT, BUDGET, "" if HT <= BUDGET else "  ** EXCEEDS BUDGET by %.2f in **" % (HT - BUDGET)))
    NM = {"Credit Utilisation Rate": "Credit\nUtilisation\nRate", "90+ Days Late": "90+ Days\nLate", "30-59 Days Past Due": "30-59 Days\nPast Due", "60-89 Days Past Due": "60-89 Days\nPast Due",
          "Open Credit Lines": "Open Credit\nLines", "Debt Ratio": "Debt\nRatio", "Monthly Income": "Monthly\nIncome", "Real Estate Loans": "Real Estate\nLoans", "Number of Dependents": "Number of\nDependents",
          "M_Income": "M_\nIncome", "M_Dependents": "M_\nDependents"}
    lab = lambda n: NM.get(n.replace(" (bin)", ""), n.replace(" (bin)", "")); IND = {"M_Income", "M_Dependents"}
    tf = plt.figure(figsize=(3, 3), dpi=100); rend = tf.canvas.get_renderer()
    def meas(s, size, wt):
        t = tf.text(0.1, 0.1, s, fontsize=size, fontweight=wt, family=FONT, linespacing=1.0); bb = t.get_window_extent(rend); t.remove(); return bb.width / 100, bb.height / 100
    def fits(size, wt):
        for n in set(list(A) + list(B)):
            w, h = meas(lab(n), size, wt)
            if n in IND:
                if w / 2 > RAD * 0.92 or h / 2 > RAD * 0.92: return False
            elif np.hypot(w / 2, h / 2) > RAD * 0.95: return False
        return True
    SIZES = [9, 8.5, 8, 7.5, 7, 6.5, 6]; choice = next(((s, wt) for wt in ("bold", "normal") for s in SIZES if fits(s, wt)), None); plt.close(tf)
    assert choice, "no label size >= 6 pt fits inside RAD=%.2f in: increase RAD or report" % RAD
    FS, WT = choice; print("node label font: %s %s %.1f pt (6 pt floor; bold tried first)" % (FONT, WT, FS))
    fig = plt.figure(figsize=(WIDTH, HT)); info = {}; tf2 = None
    for tag, edges, pos, h0, hh in (("a", E, A, HL + hB, hA), ("b", SE, B, HL, hB)):
        ax = fig.add_axes([0, h0 / HT, 1, hh / HT]); ax.set_xlim(0, WIDTH); ax.set_ylim(0, hh); ax.axis("off")
        bl = {u for u, v in edges if v == "Default"} | {v for u, v in edges if u == "Default"}; P = {}; labbox = {}
        for n, (x, y) in pos.items():
            if n == "Default": p = Circle((x, y), RAD, fc="gold", ec="black", lw=1.2, zorder=3)
            elif n in IND: p = Rectangle((x - RAD, y - RAD), 2 * RAD, 2 * RAD, fc="lightgray", ec="black", lw=0.8, zorder=3)
            elif n in bl: p = Circle((x, y), RAD, fc="lightblue", ec="darkorange", lw=2.2, zorder=3)
            else: p = Circle((x, y), RAD, fc="lightblue", ec="black", lw=0.8, zorder=3)
            ax.add_patch(p); P[n] = p; t = ax.text(x, y, lab(n), ha="center", va="center", fontsize=FS, fontweight=WT, family=FONT, zorder=4, linespacing=1.0); t.set_gid("node")
            w, h = meas(lab(n), FS, WT) if False else (0, 0)
        drawn = set()
        for u, v in edges:
            ind = u in IND; drawn.add((u, v))
            ax.add_patch(FancyArrowPatch(pos[u], pos[v], patchA=P[u], patchB=P[v], arrowstyle="-|>", mutation_scale=6, connectionstyle="arc3,rad=0.08", color="dimgray" if ind else "gray", lw=1.1 if ind else 0.6, ls="--" if ind else "-", shrinkA=0, shrinkB=0, zorder=2))
        assert drawn == set(edges) and len(drawn) == len(edges), "drawn edge set differs from stored structure"
        ax.text(0.005, 0.995, "(%s)" % tag, transform=ax.transAxes, fontweight="bold", fontsize=9, va="top", family=FONT)
        ns = list(pos); mind = min(dist(pos[a], pos[b]) for i, a in enumerate(ns) for b in ns[i + 1:])
        under = 0
        for u, v in edges:
            for tt in np.linspace(0.05, 0.95, 60):
                p = (pos[u][0] * (1 - tt) + pos[v][0] * tt, pos[u][1] * (1 - tt) + pos[v][1] * tt)
                if any(dist(p, pos[m]) < RAD for m in ns if m not in (u, v)): under += 1
        pl = (0.0, hh - 0.3, 0.3, hh); lblpanel = sum(pl[0] < x + RAD and pl[2] > x - RAD and pl[1] < y + RAD and pl[3] > y - RAD for x, y in pos.values())
        info[tag] = dict(nodes=len(pos), edges=len(edges), crossings=int(crossings(edges, pos)), min_centre_distance=round(mind, 3), edge_samples_under_other_nodes=under, nodes_touching_panel_letter=int(lblpanel))
    la = fig.add_axes([0, 0, 1, HL / HT]); la.set_xlim(0, WIDTH); la.set_ylim(0, HL); la.axis("off"); y = HL / 2
    for x, kind, t in ((0.2, "d", "Default"), (1.0, "b", "Markov blanket of Default"), (3.0, "o", "other variable"), (4.3, "s", "missingness indicator (dashed edges)")):
        if kind == "d": la.add_patch(Circle((x, y), 0.07, fc="gold", ec="black", lw=1.0))
        elif kind == "b": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="darkorange", lw=1.6))
        elif kind == "o": la.add_patch(Circle((x, y), 0.07, fc="lightblue", ec="black", lw=0.8))
        else: la.add_patch(Rectangle((x - 0.07, y - 0.07), 0.14, 0.14, fc="lightgray", ec="black", lw=0.8))
        la.text(x + 0.13, y, t, fontsize=7.5, va="center", family=FONT)
    print("collision / layout check:", json.dumps(info))
    print("smallest node label font used: %.1f pt | final height %.2f in" % (FS, HT))
    save_fig(fig, "sel_paper_fig1_structures", [])
fig1_inside()
ims = [(t, Image.open(p).convert("RGB")) for t, p in (("labels BESIDE (current)", R + "/selection/figures/sel_paper_fig1_structures_beside.png"), ("labels INSIDE (new)", R + "/selection/figures/sel_paper_fig1_structures.png"))]
ims = [(t, i.resize((1500, int(i.height * 1500 / i.width)))) for t, i in ims]; sheet = Image.new("RGB", (3020, max(i.height for _, i in ims) + 40), "white"); d = ImageDraw.Draw(sheet)
for k, (t, i) in enumerate(ims): sheet.paste(i, (k * 1510, 36)); d.text((k * 1510 + 4, 8), t + " (equal width)", fill="black")
sheet.save(FIGW + "fig1_contact_sheet_inside_vs_beside.png"); print("contact sheet saved: fig1_contact_sheet_inside_vs_beside.png")